# Week 3, Day 2: Batch Geoprocessing

## Today

By the end of this session, you will be able to:

- Explain batch-processing advantages
- List feature classes, rasters, and fields with `arcpy.ListFeatureClasses`, `arcpy.ListRasters`, and `arcpy.ListFields`, including wildcard and datatype filters
- Loop `arcpy.analysis.Buffer` over every feature class in a workspace, building each output name from its input name
- Guard a batch run with `arcpy.Exists` and `arcpy.env.overwriteOutput`, and nest two loops to run several distances across several layers

## The promist of batch processing: one tool, run on everything

Last session focused on lists-and-loops using pure Python (we didn't use any arcpy/GIS tools). Today, we're adding `arcpy` to the mix.

### Today's requirements

- ArcGIS Pro (and a notebook in it)
- The `part1` data directory from GitHub
- A setup workspace

**Predict before you run:** You're about to set the workspace and turn on `overwriteOutput`, same as last week. BEFORE YOU DO, recall what the setting `overwriteOutput = True` changes when re-running a tool call whose output name already exists?

In [ ]:
import arcpy

workspace_location = "C:/Users/pbitterman/Desktop/week03_02/" # your workspace location goes here - change it as appropriate
arcpy.env.workspace = workspace_location
arcpy.env.overwriteOutput = True

**Expected result?** nothing prints. `arcpy.env.workspace` points every unqualified filename in this notebook at your specified location.

`overwriteOutput = True` means a tool call that writes to a name that already exists will replace it instead of raising an error.

## Listing feature classes

`arcpy.ListFeatureClasses()` returns every feature class arcpy can see in the current workspace, formatted as a Python list of names. This list data structure is the same we used during last session.

**Predict before you run:** The `part1` data folder holds three shapefiles at its root: `ohio_counties.shp`, `ohio_state_parks.shp`, `portage_adj_munis.shp`. It also contains `streams_portage.geojson` and a `.csv` table. 

Guess how many names `arcpy.ListFeatureClasses()` returns.

In [ ]:
feature_classes = arcpy.ListFeatureClasses()

print(feature_classes)
print(len(feature_classes))

**What happened?**

- What did Python return?
- What DIDN'T it return?
- Why?

## Wildcard filters

`ListFeatureClasses()` can also take an optional wildcard as its first argument. Here, it uses the same `*` pattern you'd type into a file search box

**Predict before you run:** Guess which of the three feature classes match the wildcard `"ohio*"`

In [ ]:
ohio_layers = arcpy.ListFeatureClasses("ohio*")
print(ohio_layers)

**Expected result:** 

- Two matches: `['ohio_counties.shp', 'ohio_state_parks.shp']` 
- `portage_adj_munis.shp` doesn't start with `"ohio"`, so it is excluded. 

Wildcards are useful if a project's workspace has dozens of files and you only want the ones from a particular data source or naming convention

## Datatype filters

The second optional argument filters by geometry type (e.g., `"Point"`, `"Polygon"`, `"Polyline"`)

**Predict before you run:** All three feature classes in this workspace happen to be polygon shapefiles — county boundaries, municipal boundaries, and park boundaries. Guess what `arcpy.ListFeatureClasses("*", "Point")` returns.

In [ ]:
point_layers = arcpy.ListFeatureClasses("*", "Point")
polygon_layers = arcpy.ListFeatureClasses("*", "Polygon")

print(f"point layers: {point_layers}")
print(f"polygon layers: {polygon_layers}")

**What happened?**

- For point layers, `None` or `[]`, depending on your arcpy version
- For polygon layers, `polygon_layers` comes back with all three names

**Your turn:**  Chain together what you've learned so far: call `arcpy.ListFeatureClasses()` with a wildcard that matches only `portage_adj_munis.shp` (not either `ohio_*.shp` file), then call `arcpy.ListFeatureClasses("*", "Polyline")` and predict the result before running — this workspace's three shapefiles are all polygons, so what should come back?


In [ ]:
## Try it here

## Listing rasters

`arcpy.ListRasters()` is `ListFeatureClasses()`'s counterpart for raster datasets. It taks the same wildcard and datatype-filter arguments, just for different file types.

**Predict before you run:** 

This week's part1 data folder has no raster datasets at all (those come later). Guess what `arcpy.ListRasters()` returns on a workspace with zero rasters.

In [ ]:
rasters = arcpy.ListRasters()
print(rasters)

**Expected result:** `None`. Is this the same as `ListFeatureClasses()`, or is it different?

## Listing fields

Instead of listing files, we can look "inside" a file and inspect its schema (filed names, types, lengths). To do so, we want to use:

`arcpy.ListFields()` 

**Predict before you run:** Guess whether a field called `NAME` shows up in `ohio_counties.shp`'s field list, and roughly how many fields total you'd expect from a Census-derived county boundary file.

In [ ]:
fields = arcpy.ListFields("ohio_counties.shp")

for field in fields:
    print(field.name, field.type)

**Expected result:** 


a printed line for every field arcpy finds, starting with shapefile standard files (object ID field, `Shape` geometry field), then followed by the file's own attribute columns: `STATEFP`, `COUNTYFP`, `COUNTYNS`, `GEOID`, `GEOIDFQ`, `NAME`, `NAMELSAD`, `LSAD`, `CLASSFP`, `MTFCC`, `CSAFP`, `CBSAFP`, `METDIVFP`, `FUNCSTAT`, `ALAND`, `AWATER`, `INTPTLAT`, `INTPTLON`.

***What types are the fields?***

**Predict before you run:** Above (with `ohio_counties.shp`) included a field named `NAME`, in ALL CAPS. 

What do you think - will `ohio_state_parks.shp` have an equivalent field, and will it be formatted the same?

***Let's try it***

In [ ]:
park_fields = arcpy.ListFields("ohio_state_parks.shp")

for field in park_fields:
    print(field.name, field.type)

**Expected result:** 

- a shorter field list than `ohio_counties.shp`
- But "name" is handled differently. In this case, as a lowercase `name` field (not `NAME`)

**The lession?**

Don't assume naming formats, field names, or even data types. Always check. And do so programmatically

**Your turn:** Compare `ohio_counties.shp`'s and `portage_adj_munis.shp`'s field lists side by side with two `ListFields()` calls.

1. Which fields are are included in EVERY shapefile, and which are specific to one file's data source? 

2. Then predict what `arcpy.ListRasters("*", "Point")` would return, given this workspace has no rasters of any kind.

## Putting this week's lessons together: looping Buffer over every feature class

Combine `ListFeatureClasses()` with last session's f-string naming pattern, then use one `Buffer` call to handle every feature class in the workspace.

**AS A CLASS, let's predict before we run:** 

- A volunteer to front???


***Let's breakdown the code***

In [ ]:
feature_classes = arcpy.ListFeatureClasses()

for fc in feature_classes:
    base_name = fc.replace(".shp", "") # What's this? How would you know? Where would you look?
    out_name = f"{base_name}_1mi.shp"
    arcpy.analysis.Buffer(fc, out_name, "1 Mile")
    print(f"buffered {fc} -> {out_name}")

**Expected result???** 

**Your turn:** Rewrite the Buffer loop above so the output name pattern is `f"{base_name}_500ft.shp"` and the distance is `"500 Feet"` instead of `"1 Mile"`. Predict the three output filenames before you run it on a lab machine.


In [ ]:
## Try it here



## "Guarding" a batch run: `Exists` vs. `overwriteOutput`

`overwriteOutput = True` (as we earlier) allows a tool to silently replace an output that already exists

In using `arcpy.Exists()`, we can do something different, including *skipping* work entirely

***When might this be useful?***

**AS A CLASS, predict before you run:** The loop above already created all three `_1mi.shp` outputs. Guess what the loop below prints if you run it again right now, with an `arcpy.Exists()` guard added.

***Another volunteer***

In [ ]:
for fc in feature_classes:
    base_name = fc.replace(".shp", "")
    out_name = f"{base_name}_1mi.shp"
    if arcpy.Exists(out_name):
        print(f"skipping {out_name} -- already exists")
        continue
    arcpy.analysis.Buffer(fc, out_name, "1 Mile")
    print(f"buffered {fc} -> {out_name}")

**Expected result:** What did we get?

**Your turn:** 
1. Manually delete or rename just **one** of the three `_1mi.shp` outputst
2. hen rerun the guarded loop above. Predict which two lines print `skipping ... -- already exists` and identify which line actually calls `Buffer` 

## Nested loops: One last bit of complexity

### (yo dawg, I put a loop inside your loop)

Let's nest two loops — one for the distance, one for the layer — and let arcpy run every combination

**AS A CLASS, let's predict before you run:** Three feature classes, three distances. Guess the total number of output files the nested loop below creates.

***A final volunteer?***

In [ ]:
buffer_distances = ["0.5 Mile", "1 Mile", "2 Miles"]
feature_classes = arcpy.ListFeatureClasses()

for distance in buffer_distances:
    distance_label = distance.replace(" ", "_").replace(".", "p")
    for fc in feature_classes:
        base_name = fc.replace(".shp", "")
        out_name = f"{base_name}_{distance_label}.shp"
        arcpy.analysis.Buffer(fc, out_name, distance)
        print(f"{out_name} created at {distance}")

**Expected result:** Did we get what we expected?

**Your turn:** Add a fourth distance to `buffer_distances` and predict the new total file count before you run the nested loop again on a lab machine.

In [ ]:
## Try it here (or just edit the above block)


## When failure happens (and it will)

A batch of 50 real files eventually contains one that breaks the pattern. Why?

- locked by another user
- a bad projection
- a typo'd name. 

If we don't handle the error, the one failure stops *the whole loop!* We'll do more in later weeks, but here's a preview


**Predict before you run:** `fcs_to_process` below includes a filename that doesn't exist in this workspace. Will the loop crashes on that file, or will it keep going and finish the other three?

In [ ]:
fcs_to_process = feature_classes + ["missing_layer.shp"]

for fc in fcs_to_process:
    try:
        base_name = fc.replace(".shp", "")
        out_name = f"{base_name}_1mi.shp"
        arcpy.analysis.Buffer(fc, out_name, "1 Mile")
        print(f"buffered {fc}")
    except arcpy.ExecuteError:
        print(f"skipped {fc} -- arcpy could not process it")

**Expected result:** What happened?

Let's talk about `try` and `except`

**Your turn:** Pick a filename you know doesn't exist in this workspace, and wrap one `Buffer` call for it in `try`/`except` — confirm your notebook keeps running instead of crashing

In [ ]:
## Try it here

## Try it yourself

Chain three of today's patterns end to end: 

1. list this workspace's feature classes
2. filter to just the ones matching a wildcard of your own choosing
3. and buffer each match by a distance you pick 

And do so by building each output name the same way the nested-loop example did

**This week's reading:** See Canvas

**Assessments:** Lab 1 due next week

**Next week's topic:** Data formats... shapefiles vs. file geodatabases vs. GeoPackage vs. GeoParquet 